# 8장 생존 자료와 Kaplan-Meier (R과 파이썬으로 하는 통계분석)

survival 패키지의 폐암 자료(lung)로 Kaplan-Meier 곡선을 그리고 군의 생존을 비교합니다. 보건통계학 기초의 파이썬 실습 7과 같은 자료입니다. e 절에서는 보건통계학 기초 7장의 예(청구 기록 4명, 12명, 약물 A·B 300명)를 계산합니다. Colab에서는 런타임 → 런타임 유형 변경에서 R을 고른 뒤 셀을 위에서부터 차례로 실행하세요.


## a. 생존 자료를 R에 알려 주기

### 셀 1. 폐암 자료 불러오기

In [ ]:
library(survival)
head(lung, 3)
table(lung$status)

### 셀 2. Surv()로 생존 자료 만들기

In [ ]:
lung$months <- lung$time / 30.4375      # 일 → 개월 (365.25 / 12)
s <- Surv(lung$months, lung$status)
head(s)
table(s[, "status"])

### 셀 3. 중앙 추적기간 (역 Kaplan-Meier)

In [ ]:
survfit(Surv(months, status == 1) ~ 1, data = lung)   # 중도절단을 사건으로 본 역 KM

## b. Kaplan-Meier 곡선과 중앙생존기간

### 셀 4. Kaplan-Meier 추정과 시점별 생존율

In [ ]:
km <- survfit(Surv(months, status) ~ 1, data = lung)
km
summary(km, times = c(6, 12, 24))

### 셀 5. 신뢰구간을 log-log 방식으로

In [ ]:
km_ll <- survfit(Surv(months, status) ~ 1, data = lung, conf.type = "log-log")
km_ll
summary(km_ll, times = c(6, 12, 24))

### 셀 6. 기본 그림으로 곡선 그리기

In [ ]:
plot(km_ll, xlab = "Months", ylab = "Survival probability",
     mark.time = TRUE, xaxt = "n")
axis(1, at = seq(0, 36, by = 6))
abline(h = 0.5, lty = 3, col = "gray50")

## c. 군별 곡선과 요약표

### 셀 7. 성별 Kaplan-Meier

In [ ]:
lung$sex_f <- factor(lung$sex, levels = 1:2, labels = c("Male", "Female"))
km_sex <- survfit(Surv(months, status) ~ sex_f, data = lung, conf.type = "log-log")
print(km_sex, rmean = 24)

### 셀 8. 논문에 옮길 성별 요약표

In [ ]:
st  <- summary(km_sex)$table           # 군별 n, 사건 수, 중앙값과 구간
s12 <- summary(km_sex, times = 12)     # 12개월 생존율과 구간
data.frame(group = levels(lung$sex_f),
           n = st[, "n.max"], deaths = st[, "events"],
           `median (95% CI)` = sprintf("%.1f (%.1f-%.1f)",
                                       st[, "median"], st[, "0.95LCL"], st[, "0.95UCL"]),
           `1-yr % (95% CI)` = sprintf("%.1f (%.1f-%.1f)",
                                       100 * s12$surv, 100 * s12$lower, 100 * s12$upper),
           check.names = FALSE, row.names = NULL)

### 셀 9. survminer로 곡선과 위험집합 표

In [ ]:
install.packages("survminer")
library(survminer)
ggsurvplot(km_sex, data = lung, conf.int = TRUE, pval = TRUE,
           risk.table = TRUE, break.time.by = 6,
           legend.title = "Sex", legend.labs = c("Male", "Female"),
           xlab = "Months", palette = c("#1f5f8b", "#c8553d"))

## d. 로그순위 검정과 그 밖의 비교

### 셀 10. 로그순위 검정

In [ ]:
lr <- survdiff(Surv(months, status) ~ sex_f, data = lung)
lr
lr$pvalue

### 셀 11. 초기 차이에 가중치를 더 준 검정 (rho = 1)

In [ ]:
survdiff(Surv(months, status) ~ sex_f, data = lung, rho = 1)

### 셀 12. 세 군 비교와 두 군씩 비교

In [ ]:
lung$ecog <- factor(ifelse(lung$ph.ecog >= 2, "2+", lung$ph.ecog))
table(lung$ecog, useNA = "ifany")
survdiff(Surv(months, status) ~ ecog, data = lung)
pairwise_survdiff(Surv(months, status) ~ ecog, data = lung,
                  p.adjust.method = "bonferroni")

### 셀 13. 24개월 제한평균생존시간(RMST)의 차이

In [ ]:
rm <- summary(km_sex, rmean = 24)$table[, c("rmean", "se(rmean)")]
rm
d  <- rm["sex_f=Female", "rmean"] - rm["sex_f=Male", "rmean"]
se <- sqrt(sum(rm[, "se(rmean)"]^2))
round(c(diff = d, lower = d - 1.96 * se, upper = d + 1.96 * se), 2)

## e. 보건통계학 기초 7장의 예

### 셀 14. 청구 기록을 time과 status로 바꾸기

In [ ]:
d <- data.frame(pt = c("A", "B", "C", "D"),          # 가상의 환자 4명 (7장 a 절의 가·나·다·라)
  index   = as.Date(c("2021-02-15", "2021-06-01", "2022-01-10", "2022-03-20")),   # 첫 처방일
  last_rx = as.Date(c("2021-11-10", "2021-09-01", "2022-04-10", "2022-12-10")),   # 마지막 처방일
  supply  = c(90, 90, 30, 90),                                                     # 마지막 처방의 공급일수
  hhf     = as.Date(c("2022-01-20", "2022-04-18", NA, NA)),                        # 심부전 입원일
  death   = as.Date(c(NA, NA, "2022-05-02", NA)))                                  # 사망일
data_end <- as.Date("2022-12-31")                      # 자료 기간 종료
d$stop <- d$last_rx + d$supply + 60                    # 중단일 = 마지막 처방 + 공급일수 + 유예 60일
d$end_at  <- pmin(d$hhf, d$death, d$stop, data_end, na.rm = TRUE)   # as-treated: 가장 이른 날
d$end_itt <- pmin(d$hhf, d$death, data_end, na.rm = TRUE)           # ITT 유사: 중단은 끝이 아님
for (a in c("at", "itt")) {
  end <- d[[paste0("end_", a)]]
  d[[paste0("time_", a)]]   <- as.numeric(end - d$index)            # 날짜 차이(일)
  d[[paste0("status_", a)]] <- as.integer(!is.na(d$hhf) & end == d$hhf)   # 입원으로 끝나면 1
}
d[, c("pt", "stop", "end_at", "time_at", "status_at", "time_itt", "status_itt")]
round(colSums(d[, c("time_at", "time_itt")]) / 365.25, 2)           # 총 인년

### 셀 15. 약물 A·B 300명의 Kaplan-Meier (7장 b 절)

In [ ]:
co <- read.csv("https://socialp-ajou.tecentriq12.workers.dev/data/ch07_cohort.csv")
km7 <- survfit(Surv(time, status) ~ drug, data = co, conf.type = "log-log")
km7
summary(km7, times = c(12, 24, 36))

### 셀 16. 신뢰구간 방법과 추적기간

In [ ]:
survfit(Surv(time, status) ~ drug, data = co)          # 기본값 conf.type = "log"
survfit(Surv(time, status == 0) ~ 1, data = co)        # 역 Kaplan-Meier: 중앙 추적기간

### 셀 17. 12명의 로그순위 검정과 순열 P (7장 c 절)

In [ ]:
time12 <- c(3, 5, 8, 12, 12, 16, 20, 25, 28, 32, 38, 44)   # 7장 표 7-1의 12명 (개월)
st12   <- c(1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0)             # 1 = 사망
drugA  <- c(0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1)             # 약물 A = 1
lr12 <- survdiff(Surv(time12, st12) ~ drugA)
lr12
idx  <- combn(12, 6)                                        # A 이름표 6개를 붙이는 924가지
chis <- apply(idx, 2, function(i) {
  g <- seq_len(12) %in% i                                   # 이 배정에서 A인 환자
  survdiff(Surv(time12, st12) ~ g)$chisq
})
c(n = length(chis), n_ge = sum(chis >= lr12$chisq - 1e-9), p = mean(chis >= lr12$chisq - 1e-9))

### 셀 18. 300명의 로그순위 검정과 층화

In [ ]:
survdiff(Surv(time, status) ~ drug, data = co)                  # 층화하지 않음
survdiff(Surv(time, status) ~ drug + strata(imdc), data = co)   # IMDC 위험군으로 층화

### 셀 19. 제한평균생존시간

In [ ]:
k12 <- survfit(Surv(time12, st12) ~ 1)
print(k12, rmean = 36)                                  # 12명의 36개월 RMST (7장 b 절)
rm <- summary(km7, rmean = 48)$table[, c("rmean", "se(rmean)")]
dd <- rm["drug=A", "rmean"] - rm["drug=B", "rmean"]
se <- sqrt(sum(rm[, "se(rmean)"]^2))
round(c(diff = dd, lower = dd - 1.96 * se, upper = dd + 1.96 * se), 2)   # 48개월 RMST 차이

## f. 과제 정답

### 셀 20. 과제 1 정답: ECOG 군별 요약표

In [ ]:
km_ecog <- survfit(Surv(months, status) ~ ecog, data = lung, conf.type = "log-log")
st  <- summary(km_ecog)$table
s12 <- summary(km_ecog, times = 12)
data.frame(group = rownames(st), n = st[, "n.max"], deaths = st[, "events"],
           `median (95% CI)` = sprintf("%.1f (%.1f-%.1f)",
                                       st[, "median"], st[, "0.95LCL"], st[, "0.95UCL"]),
           `1-yr % (95% CI)` = sprintf("%.1f (%.1f-%.1f)",
                                       100 * s12$surv, 100 * s12$lower, 100 * s12$upper),
           check.names = FALSE, row.names = NULL)

### 셀 21. 과제 2 정답: ECOG로 층화한 로그순위 검정

In [ ]:
survdiff(Surv(months, status) ~ sex_f + strata(ecog), data = lung)
survdiff(Surv(months, status) ~ sex_f, data = lung, subset = !is.na(ecog))

### 셀 22. 과제 3 정답: 사건 변수를 잘못 만들면

In [ ]:
lung$dead_wrong <- as.integer(lung$status > 0)    # 실수: 모두 1이 된다
table(lung$dead_wrong)
survfit(Surv(months, dead_wrong) ~ 1, data = lung)